In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


In [21]:
iris = load_iris()

X = iris.data
y = iris.target

print("X shape:", X.shape)
print("y shape:", y.shape)

print("Classes:", iris.target_names)

X shape: (150, 4)
y shape: (150,)
Classes: ['setosa' 'versicolor' 'virginica']


In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(120, 4)
(30, 4)


In [23]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [24]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [25]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

In [26]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [27]:
class IrisModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(4, 16)
        self.layer2 = nn.Linear(16, 8)
        self.output = nn.Linear(8, 3)

    def forward(self, x):

        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        x = self.output(x)

        return x

In [28]:
model = IrisModel()

model = model.to(device)

print(model)

IrisModel(
  (layer1): Linear(in_features=4, out_features=16, bias=True)
  (layer2): Linear(in_features=16, out_features=8, bias=True)
  (output): Linear(in_features=8, out_features=3, bias=True)
)


In [29]:
criterion = nn.CrossEntropyLoss()

In [30]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [31]:
epochs = 100

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        # Move data to same device as model
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(X_batch)

        # Calculate loss
        loss = criterion(outputs, y_batch)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:

        avg_loss = total_loss / len(train_loader)

        print(
            f"Epoch [{epoch+1}/{epochs}], "
            f"Loss: {avg_loss:.4f}"
        )

Epoch [10/100], Loss: 0.9570
Epoch [20/100], Loss: 0.5950
Epoch [30/100], Loss: 0.3908
Epoch [40/100], Loss: 0.2616
Epoch [50/100], Loss: 0.1905
Epoch [60/100], Loss: 0.1490
Epoch [70/100], Loss: 0.1067
Epoch [80/100], Loss: 0.0893
Epoch [90/100], Loss: 0.0729
Epoch [100/100], Loss: 0.0676


In [32]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)

        predictions = torch.argmax(outputs, dim=1)

        total += y_batch.size(0)

        correct += (predictions == y_batch).sum().item()

accuracy = correct / total

print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 96.67%


In [33]:
new_flower = [
    [5.1, 3.5, 1.4, 0.2]
]

In [34]:
new_flower = scaler.transform(new_flower)

In [35]:
new_flower = torch.tensor(
    new_flower,
    dtype=torch.float32
)

new_flower = new_flower.to(device)

In [36]:
model.eval()

with torch.no_grad():

    output = model(new_flower)

    prediction = torch.argmax(output, dim=1)

print("Predicted class:", prediction.item())
print("Flower:", iris.target_names[prediction.item()])

Predicted class: 0
Flower: setosa
